In [ ]:
import sys
import os

import tensorflow as tf

os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('variational_ae.py'), '..')))

from tensorflow.keras.datasets import mnist
from variational_ae import VariationalAutoencoder
import numpy as np
from sklearn.model_selection import train_test_split
import h5py

In [ ]:
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"         # Keeps GPU order consistent
os.environ["CUDA_VISIBLE_DEVICES"] = "0"               # Makes only GPU 0 visible (useful even with 1 GPU)

gpus = tf.config.experimental.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)  # Prevents TF from using all GPU memory at once
    except RuntimeError as e:
        print("Error: ", e)
        exit(-1)

In [ ]:
with h5py.File('Dataset/log_spec_data_dataset.h5', 'r') as h5f:
    log_spec_data_train = h5f['train'][:]
    log_spec_data_labels = h5f['label'][:]

In [ ]:
log_spec_data_train = log_spec_data_train[..., np.newaxis]

log_spec_x_train, log_spec_x_val = train_test_split(log_spec_data_train, test_size=0.05, random_state=42)

In [ ]:
print("Log Spec Train shape:", log_spec_x_train.shape)
print("Log Spec Validation shape:", log_spec_x_val.shape)

In [ ]:
LEARNING_RATE = 0.0005
BATCH_SIZE = 8
EPOCHS = 150

In [ ]:
input_shape = log_spec_x_train.shape[1:]
latent_space_dim = 2
decoder_out_filter = 1

In [ ]:
# Hyperparameters for the Variational Autoencoder
recon_weight = 100000.0  # Weight for the reconstruction loss.
beta = 1.0  # Weight for the KL divergence loss.

In [ ]:
autoencoder = VariationalAutoencoder(input_shape, latent_space_dim, decoder_out_filter, recon_weight, beta, conv_layers_config=[
    {'filters': 64, 'kernel_size': (5, 5), 'strides': (2, 2)},
    {'filters': 64, 'kernel_size': (3, 3), 'strides': (1, 1)},
    
    {'filters': 128, 'kernel_size': (3, 3), 'strides': (1, 1)},
    {'filters': 128, 'kernel_size': (3, 3), 'strides': (1, 1)},
    
    {'filters': 256, 'kernel_size': (3, 3), 'strides': (1, 1)},
    {'filters': 256, 'kernel_size': (3, 3), 'strides': (1, 1)},
    
    {'filters': 384, 'kernel_size': (3, 3), 'strides': (1, 1)},
    
    {'filters': 512, 'kernel_size': (3, 3), 'strides': (1, 1)},
    
    {'filters': 256, 'kernel_size': (3, 3), 'strides': (1, 1)},
])

In [ ]:
autoencoder.compile(learning_rate=LEARNING_RATE)

In [ ]:
autoencoder.summary()

In [ ]:
autoencoder.fit(
    x=log_spec_x_train,
    y={"reconstruction": log_spec_x_train}, # Autoencoders typically use the same data for input and output
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(log_spec_x_val, {"reconstruction": log_spec_x_val}), # Validation data for monitoring
    shuffle=True
)

In [ ]:
autoencoder.save_all()  # Save the trained model